# Collapse simulation & experimental magnetization onto a characteristic-field axis

**Goal.** Put the simulation and experimental per-particle moment curves on one *unit-free*
reduced field so they overlay even when their susceptibilities differ.

**Why not a temperature.** ξ = μ·B/kT depends only on the ratio **3χ/M_sat** (since
χ = μ₀nμ²/3kT and M_sat = nμ). For ~3 µm athermal carbonyl iron μ is enormous, so fitting
(μ, kT) reconstructs that finite ratio out of two huge/uncertain numbers — it is
ill-conditioned and hyper-sensitive to kT (the curve slides as B\*=kT/μ while the raw data vs H
is fixed), and it cannot compare systems whose energy differs (real T vs sim kT_p vs none).

**Instead** reduce **each curve by its own measured susceptibility χ and saturation M_sat**:

    y = M / M_sat ,    α = 3·χ·H / M_sat    (= the MMF2 Langevin argument; μ₀ cancels)

α is **order 1**, identical in form for SI experiment and sim units, and the mechanism-specific
energy is upstream of χ — so **no temperature is ever chosen**. χ is *measured* (MMF2 fit to the
near-saturation approach), not assumed. The characteristic field H₀ = M_sat/χ replaces the old
`K_BT`.

**Field convention.** The simulation uses the **external** field H_ext = B_applied/μ₀ (the
internal/demag variant was dropped — external collapses the two data sets better). The
experimental sheets are recorded against their own internal field (column A), which is what the
Gasper data provides.

Pipeline: helpers → MMF2 (susceptibility model) → load every sim & experimental curve, measure χ
per curve → reduce to α → summary plots (moment & magnetization) + standalone legends.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook

from pyanal.ARay import ARay

# ----------------------------------------------------------------- constants --
espresso_prefactor = 1.0
mu_0_SI  = 4 * np.pi * 1e-7          # T m / A
mu_0_sim = 4 * np.pi * espresso_prefactor

dens_iron_SI      = 7874            # kg/m^3
dens_elastomer_SI = 1000            # kg/m^3

## 1. Configuration

Experiment = the Gasper CIP sheets; simulation = the `SM`, HEIGHT-10 ARay slice. χ is **measured
per curve** from the near-saturation approach (the `*_SAT_MIN` thresholds set how near), not fitted
as a temperature.

In [ ]:
from repo_paths import DATA_DIR   # repository-relative; set $MAE_DATA_DIR to read the dataset from elsewhere
import os
# ---- experiment ----
EXP_FILE    = os.path.join(DATA_DIR, "magnetization_curves/gasper-magnetization.xlsx")
EXP_COLS    = ["A", "B"]
EXP_HEADERS = ["H internal [kA/m]", "M [kA/m]"]   # column A is the INTERNAL field
SIZE_CIP    = 3e-6                  # m, particle diameter
M_S_EXP     = 1669e3                             # A/m  -- real bulk saturation (1669 kA/m)
V_PART      = np.pi / 6 * SIZE_CIP ** 3          # m^3  -- single-particle volume
MU_SAT_EXP  = V_PART * M_S_EXP                   # A m^2 -- real saturation moment (all curves)

# ---- simulation ----
SIM_ARAY    = os.path.join(DATA_DIR, "figs/data_aray/SM--HEIGHT-10-full.h5")
TS, HEIGHT  = 500, 10
SIZE_SIM    = 1.0
MU_MAX_SIM  = 1.0                   # per-particle saturation moment (sim units) -> m_sat = 1

## 2. Helper functions

Unit conversions, number density, the Excel loader, and a robust ascending/descending
hysteresis-branch splitter.

In [ ]:
def mu_0_units(units):
    if units == "sim":
        return mu_0_sim
    if units == "SI":
        return mu_0_SI
    raise ValueError(f"Invalid units: {units}")

# ---- field / moment / magnetization conversions ----
def H_from_B(B, M=0.0, units="SI"):
    return B / mu_0_units(units) - M

def B_from_H(H, M=0.0, units="SI"):
    return mu_0_units(units) * (H + M)

def magnetization_from_moment(m, n):
    return n * m

def moment_from_magnetization(M, n):
    return M / n

def volume_fraction_from_weight_percentage(wp, density_particle, density_elastomer):
    a = wp / density_particle
    b = (100 - wp) / density_elastomer
    return a / (a + b)

def N_V(particle_size, volume_fraction):
    single_part_volume = np.pi / 6 * particle_size ** 3
    return volume_fraction / single_part_volume

# ---- Excel loader ----
def col_to_idx(col):
    col = col.upper()
    idx = 0
    for ch in col:
        idx = idx * 26 + (ord(ch) - ord('A') + 1)
    return idx - 1

def load_excel_columns(path, columns, headers=None, sheet=None):
    wb = load_workbook(path, data_only=True)
    if isinstance(sheet, int):
        ws = wb[wb.sheetnames[sheet]]
    elif isinstance(sheet, str):
        ws = wb[sheet]
    else:
        ws = wb.active
    rows = list(ws.values)
    header = rows[0]
    start = 1 if (headers and all(h in header for h in headers)) else 0
    idx = [col_to_idx(c) if isinstance(c, str) else c for c in columns]
    out = [[] for _ in idx]
    for row in rows[start:]:
        for i, j in enumerate(idx):
            out[i].append(row[j])
    return tuple(np.array(c, float) for c in out)

# ---- hysteresis: indices of the ascending / descending branches ----
def split_hysteresis_indices(H, smooth_window=3):
    H = np.asarray(H, dtype=float)
    n = len(H)
    dH = np.diff(H)
    if smooth_window > 1:
        kernel = np.ones(smooth_window) / smooth_window
        dH_smooth = np.convolve(dH, kernel, mode="same")
    else:
        dH_smooth = dH.copy()
    signs = np.sign(dH_smooth)
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    if signs[0] == 0:
        signs[0] = signs[signs != 0][0] if np.any(signs != 0) else 1
    point_signs = np.empty(n)
    point_signs[:-1] = signs
    point_signs[-1] = signs[-1]
    asc_idx = np.where(point_signs > 0)[0]
    desc_idx = np.where(point_signs < 0)[0]
    return asc_idx, desc_idx

## 3. MMF2 model, susceptibility measurement & the reduced field

MMF2 is the mean-field magnetization model; we use it to **measure** each curve's susceptibility
χ from its near-saturation approach (where the point-dipole / mean-field picture is valid). The
reduced field α = 3χH/M_sat is exactly the MMF2 Langevin argument — dimensionless,
unit-system independent, order 1 (α~3 marks the onset of saturation).

In [ ]:
# ---- MMF2 mean-field magnetization model + susceptibility measurement -------------------
# MMF2(B, u_max, Xi_L, n) returns the magnetization M(B) of n magnetizable particles with
# saturation moment u_max and Langevin susceptibility Xi_L; per-particle moment = M/n.
from scipy.optimize import curve_fit

def langevin_func(x):
    x = np.asarray(x, dtype=float)
    small = np.abs(x) < 1e-6
    xs = np.where(small, 1.0, x)
    L = 1.0 / np.tanh(xs) - 1.0 / xs
    return np.where(small, 0.0, L)

def derivative_langevin_func(x):
    x = np.asarray(x, dtype=float)
    small = np.abs(x) < 1e-6
    xs = np.where(small, 1.0, x)
    d = 1.0 / xs ** 2 - 1.0 / np.sinh(xs) ** 2
    return np.where(small, 0.0, d)

def langevin(B, u_max, Xi_L, n, units="sim"):
    mu_0_M_max = mu_0_units(units) * n * u_max
    alpha = 3 * Xi_L / mu_0_M_max * B
    alpha = np.sign(alpha) * np.clip(np.abs(alpha), 1e-8, None)
    return n * u_max * langevin_func(alpha)

def dlangevin_dH(B, u_max, Xi_L, n, units="sim"):
    mu_0_M_max = mu_0_units(units) * n * u_max
    alpha = 3 * Xi_L / mu_0_M_max * B
    alpha = np.sign(alpha) * np.clip(np.abs(alpha), 1e-8, None)
    return 3 * Xi_L * derivative_langevin_func(alpha)

def B_eff(B, u_max, Xi_L, n, units="sim"):
    return B + (mu_0_units(units) / 3 * langevin(B, u_max, Xi_L, n=n, units=units)) \
             * (1 + dlangevin_dH(B, u_max, Xi_L, n=n, units=units) / 48)

def MMF2(B, u_max, Xi_L, n, units="sim"):
    B_e = B_eff(B, u_max, Xi_L, n=n, units=units)
    return langevin(B_e, u_max, Xi_L, n=n, units=units)

def fit_chi_mmf2(H, y, n, u_max, units="sim", sat_min=0.95, min_pts=3):
    # Fit MMF2's susceptibility Xi_L (== chi) to the near-saturation points (y > sat_min), where
    # the approach to saturation (~1 - mu_0*M_sat/(3*Xi_L*B)) pins Xi_L. If too few points reach
    # sat_min (e.g. the sim tops out at ~0.886), fall back to the top `min_pts` points.
    # Returns (Xi_L, norm_rms) with norm_rms the fit residual in units of u_max.
    H = np.asarray(H, dtype=float)
    y = np.asarray(y, dtype=float)
    B = B_from_H(H, units=units)
    m_data = y * u_max                                  # per-particle moment
    mask = y > sat_min
    if mask.sum() < min_pts:
        idx = np.argsort(y)[-min(min_pts, len(y)):]
        mask = np.zeros(len(y), dtype=bool)
        mask[idx] = True
    (Xi_L,), _ = curve_fit(
        lambda Bx, XiL: MMF2(Bx, u_max, XiL, n, units) / n,
        B[mask], m_data[mask], p0=[1.0], bounds=([0.0], [np.inf]), maxfev=10000)
    pred = MMF2(B[mask], u_max, Xi_L, n, units) / n
    norm_rms = float(np.sqrt(np.mean((m_data[mask] - pred) ** 2)) / u_max)
    return Xi_L, norm_rms

def reduced_field(H, chi, M_sat):
    # Characteristic-field reduction: alpha = 3*chi*B/(mu_0*M_sat) = 3*chi*H/M_sat.
    # mu_0 cancels -> dimensionless and unit-system independent (exp SI & sim share one axis,
    # no kT). It is exactly the MMF2 Langevin argument; alpha~3 is the onset of saturation.
    return 3.0 * chi * H / M_sat

## 4. Load every curve onto the characteristic-field axis

Every simulation curve (DENS × K) and every Gasper sheet (wt%). Each curve is reduced by its
**own** measured susceptibility χ (MMF2 near-saturation fit) → α = 3χH/M_sat, so curves of
different susceptibility collapse with NO shared temperature and NO kT.

All sheets are the SAME material, anchored to one real bulk saturation `M_S_EXP` and one real
saturation moment `MU_SAT_EXP` = V_part·M_S_EXP. The simulation field is the **external** field.

In [ ]:
# === Load all curves and measure chi ==================================================
from pyanal.plot import sample_color_map, sample_marker_map, A_linestyle_map, A_markerfill_map
from mae_analysis.config import A_LS_POOL

EXP_VARIANTS = [(wp, k) for wp in (70, 75, 80) for k in ("soft",)]
SIM_VARIANTS = [(d, k) for d in (0.20, 0.25, 0.30) for k in ("hard", "soft")]

EXP_UP        = True   # which hysteresis branch drives the display (True=ascending)
EXP_SHOW_BOTH = True   # plot both branches (False = only the EXP_UP branch)
EXP_SAT_MIN   = 0.95   # near-saturation mask for the experiment chi fit
SIM_SAT_MIN   = 0.75   # sim tops out ~0.886 -> lower threshold (fit_chi_mmf2 falls back to top pts)

def load_experiment_sheet(weight_p, k_gasper, up=EXP_UP):
    # (alpha, y, y_mag, chi) for one Gasper sheet & branch. alpha = 3*chi*H_int/M_sat with chi
    # measured by an MMF2 near-saturation fit; y = m / MU_SAT_EXP (real per-particle saturation).
    # The sheets are recorded against their internal field (column A) -- that is the raw datum.
    H, M = load_excel_columns(EXP_FILE, EXP_COLS, EXP_HEADERS, sheet=f"{weight_p}% {k_gasper}")
    H, M = H * 1e3, M * 1e3                          # kA/m -> A/m
    asc, dsc = split_hysteresis_indices(H)
    idx = asc if up else dsc                          # chosen branch
    H, M = H[idx], M[idx]
    pos = H > 0
    H_int, M = H[pos], M[pos]
    vol_frac = volume_fraction_from_weight_percentage(weight_p, dens_iron_SI, dens_elastomer_SI)
    n_sheet  = N_V(SIZE_CIP, vol_frac)
    M_sat    = vol_frac * M_S_EXP                     # composite saturation (= n_sheet*MU_SAT_EXP)
    y        = moment_from_magnetization(M, n_sheet) / MU_SAT_EXP    # normalized per-particle moment
    chi, _   = fit_chi_mmf2(H_int, y, n_sheet, MU_SAT_EXP, "SI", EXP_SAT_MIN)
    alpha    = reduced_field(H_int, chi, M_sat)
    M_abs    = magnetization_from_moment(y * MU_SAT_EXP, n_sheet)             # native n
    M_s      = magnetization_from_moment(MU_SAT_EXP,     N_V(SIZE_CIP, 1.0))  # fully packed
    y_mag    = M_abs / M_s
    return dict(weight_p=weight_p, k=k_gasper, alpha=alpha, y=y, y_mag=y_mag, chi=chi, up=up)

def load_simulation_curve(data, dens, k, ts=TS, height=HEIGHT):
    # (H_ext, y, y_mag) for one ARay slice (sim units). The ARay `H` coordinate is espresso's
    # applied field = the external B, so H_ext = B_ext/mu_0_sim.
    sl    = data.get(ts=ts, DENS=dens, K=k, HEIGHT=height)
    mz    = sl["m_z"]
    H     = np.array([h for h in sl.coords("H") if mz.get(H=h).is_set()], dtype=float)
    m     = mz.get_set_entries().array.astype(float)
    order = np.argsort(H)
    B = H[order]
    m_z   = m[order]
    y     = m_z / MU_MAX_SIM
    M     = magnetization_from_moment(m_z, N_V(SIZE_SIM, dens))
    M_s   = magnetization_from_moment(MU_MAX_SIM, N_V(SIZE_SIM, 1.0))   # fully packed (sim units)
    y_mag = M / M_s                                              # nondim magnetization (= dens*y)
    return dict(dens=dens, k=k, y=y, y_mag=y_mag, H_ext=H_from_B(B, units="sim"))

# load everything once -- experiments: both branches when EXP_SHOW_BOTH, else only the chosen one
EXP_BRANCHES = [True, False] if EXP_SHOW_BOTH else [EXP_UP]
exp_curves = [load_experiment_sheet(wp, k, up=u)
              for wp, k in EXP_VARIANTS for u in EXP_BRANCHES]
_aray      = ARay.from_file(SIM_ARAY)
sim_curves = [load_simulation_curve(_aray, d, k) for d, k in SIM_VARIANTS]

# per-curve susceptibility -> reduced field alpha (external field, no shared temperature)
for c in sim_curves:
    n_c      = N_V(SIZE_SIM, c["dens"])
    M_sat    = n_c * MU_MAX_SIM
    chi, _   = fit_chi_mmf2(c["H_ext"], c["y"], n_c, MU_MAX_SIM, "sim", SIM_SAT_MIN)
    c["chi"]   = chi
    c["alpha"] = reduced_field(c["H_ext"], chi, M_sat)

# --- styling: reproduce mae_analysis surface "magnetic moment" encoding -----------------
# Sim:  K -> colour family (hard=Oranges, soft=Blues), DENS -> shade + marker, A="SM" -> "--"/open.
# Exp:  distinct families (soft=Greens, hard=Purples), wt% -> shade + marker, solid/filled.
SIM_DENS = [0.20, 0.25, 0.30]
EXP_WP   = [70, 75, 80]
sim_key  = lambda k, dens: f"{dens:.2f}-{k}"
exp_key  = lambda k, wp:   f"{wp}-{k}"

sim_color  = sample_color_map(["hard", "soft"], SIM_DENS, sim_key)             # default Oranges/Blues
sim_marker = sample_marker_map(["hard", "soft"], SIM_DENS, sim_key)
exp_color  = sample_color_map(["hard", "soft"], EXP_WP, exp_key,
                              k_cmaps={"hard": "Purples", "soft": "Greens"})
exp_marker = sample_marker_map(["hard", "soft"], EXP_WP, exp_key)
SIM_LS     = A_linestyle_map(["SMfl", "SM"], linestyles=A_LS_POOL)["SM"]        # "--"
SIM_FILLED = A_markerfill_map(["SMfl", "SM"])["SM"]                              # False (open)

print("=== per-curve susceptibility chi (MMF2 near-saturation fit) -> alpha = 3*chi*H/M_sat ===")
for c in sim_curves:
    print(f"    sim DENS={c['dens']:.2f} K={c['k']:<4}  chi={c['chi']:.4g}")
for c in exp_curves:
    print(f"    exp {c['weight_p']}% {c['k']:<4} up={c['up']}  chi={c['chi']:.4g}")

## 5. Summary plots & standalone legends

Two figures on the same α axis and styling:

* **moment** `y = <m>/m_inf` — normalized, curves collapse near saturation;
* **magnetization** `y_mag = M/M_s` — curves spread by loading (sim ~DENS, exp ~vol_frac).

Sim = markers + a smooth MMF2 fit (dashed) showing the approach to saturation; exp = markers only
(descending branch darkened). Legends are saved separately in three column groupings.

In [ ]:
# === Summary plots: moment AND magnetization, all curves on the characteristic-field axis ===
from matplotlib.lines import Line2D
from pyanal.plot import save_plot_simple, darken_color
from mae_analysis.legends import save_frameless_legend

fontsize_ticks  = 14
fontsize_legend = 11
fontsize_label  = 22

markersize_1 = 8

out_dir = os.path.join(DATA_DIR, "magnetization_curves/tmp")

# (curve-dict key, y-axis label, ylim, filename stem)
Y_SPECS = [("y",     r"$\left<m\right> / m_{\infty}$", [-0.05, 1.05], "tmp_collapse"),
           ("y_mag", r"$M / M_{s}$",                   None,          "tmp_collapse_mag")]

# smooth MMF2 dashed line is drawn across the full alpha range spanned by the experiments
alpha_max = max(c["alpha"].max() for c in exp_curves)
M_s_sim   = magnetization_from_moment(MU_MAX_SIM, N_V(SIZE_SIM, 1.0))   # fully-packed sim saturation

# --- build smooth MMF2 curves per sim curve using its measured chi (section 4) -> alpha grid
print("=== smooth MMF2 lines on the alpha axis (chi measured per curve in section 4) ===")
sim_fits = []
for c in sim_curves:
    n_sim  = N_V(SIZE_SIM, c["dens"])
    M_sat  = n_sim * MU_MAX_SIM
    chi    = c["chi"]
    H_grid = np.linspace(0, alpha_max * M_sat / (3 * chi), 600)   # alpha in [0, alpha_max]
    M_grid = MMF2(B_from_H(H_grid, units="sim"), MU_MAX_SIM, chi, n_sim, "sim")
    sim_fits.append(dict(c=c, chi=chi,
                         alpha=reduced_field(H_grid, chi, M_sat),
                         y=M_grid / n_sim / MU_MAX_SIM,           # normalized moment (-> 1)
                         y_mag=M_grid / M_s_sim))                 # nondim magnetization (-> dens)
    print(f"    DENS={c['dens']:.2f} {c['k']:<4}  chi={chi:.4g}")

for ykey, ylabel, ylim_, stem in Y_SPECS:
    fig, ax = plt.subplots()

    # experimental curves: markers-only, soft=Greens / hard=Purples; descending branch darkened
    for c in exp_curves:
        key   = exp_key(c["k"], c["weight_p"])
        color = exp_color[key]
        if EXP_SHOW_BOTH and not c["up"]:        # reverse (descending) branch -> darker
            color = darken_color(color)
        o = np.argsort(c["alpha"])
        ax.plot(c["alpha"][o], c[ykey][o], linestyle="none",
                color=color, marker=exp_marker[key], ms=4,
                label=f"exp {c['weight_p']}% {c['k']}")

    # simulation: markers only (data) + smooth MMF2 fit (dashed) showing approach to saturation
    for c in sim_curves:
        key   = sim_key(c["k"], c["dens"])
        o     = np.argsort(c["alpha"])
        ax.plot(c["alpha"][o], c[ykey][o], linestyle="none",
                color=sim_color[key], marker=sim_marker[key], ms=markersize_1,
                markerfacecolor=sim_color[key] if SIM_FILLED else "white",
                label=f"sim DENS={c['dens']:.2f} {c['k']}")
    for fit in sim_fits:
        key = sim_key(fit["c"]["k"], fit["c"]["dens"])
        ax.plot(fit["alpha"], fit[ykey], linestyle=SIM_LS, color=sim_color[key], lw=1.3)

    save_plot_simple(
        num=fig.number,
        filename=f"{out_dir}/{stem}_external",
        format=".png",
        xlabel=r"$\alpha = 3\chi H / M_{s}$",
        ylabel=ylabel,
        ylim=ylim_,
        fontsize_label=fontsize_label, fontsize_ticks=fontsize_ticks,
        fontsize_legend=fontsize_legend,
        axes_kw=None,
        dpi=600,
        show=True)
    print(f"[{stem}] saved -> {out_dir}/{stem}_external.png")

# --- standalone legends, mae_analysis-ordered (3 column-grouping variants) ----------------
# "model" axis = sim / exp (linestyle + fill + colour set); K = hard/soft = colour family;
# varying = DENS (sim) / wt% (exp), ascending so each column reads light->dark.
# Sim handle = MMF2 dashed line + marker; exp handle = markers-only (ls="none").
GROUPS = ["sim", "exp"]            # flip to ["exp", "sim"] to reorder
KS     = ["hard", "soft"]
group_spec = {
    "sim": dict(values=SIM_DENS, color=sim_color, marker=sim_marker, key=sim_key,
                ls=SIM_LS, filled=SIM_FILLED, label=lambda k, v: f"SM {v:.2f}-{r"$K_{\text{high}}$" if k == "hard" else r"$K_{\text{low}}$"}"),
    "exp": dict(values=EXP_WP, color=exp_color, marker=exp_marker, key=exp_key,
                ls="none", filled=True, label=lambda k, v: f"exp {volume_fraction_from_weight_percentage(v, 7874, 1000):.2f} ({v}% mass)"),
}

handle_of = {}
for g in GROUPS:
    s = group_spec[g]
    for k in KS:
        for v in s["values"]:                          # ascending -> light->dark down a column
            key = s["key"](k, v)
            handle_of[(g, k, v)] = Line2D(
                [0], [0], color=s["color"][key], marker=s["marker"][key],
                markerfacecolor=s["color"][key] if s["filled"] else "white",
                linestyle=s["ls"], label=s["label"](k, v))

# matplotlib fills legends column-major -> handle order + ncols define the grid
order_mk = [(g, k, v) for g in GROUPS for k in KS for v in group_spec[g]["values"]]
order_km = [(g, k, v) for k in KS for g in GROUPS for v in group_spec[g]["values"]]
n_models, n_K = len(GROUPS), len(KS)
legend_variants = [("bymodelK", order_mk, n_models * n_K),   # cols: (sim,exp) x (hard,soft)
                   ("byK",      order_km, n_K),              # cols: hard | soft (sim over exp)
                   ("bymodel",  order_mk, n_models)]         # cols: sim | exp (hard over soft)
for vname, order, ncols in legend_variants:
    save_frameless_legend(out_dir, f"{out_dir}/tmp_collapse_legend-{vname}.png",
                          [handle_of[key] for key in order], ncols, dpi_=600)
    print(f"legend  saved -> {out_dir}/tmp_collapse_legend-{vname}.png  (ncols={ncols})")